In [ ]:
# Path configuration
from pathlib import Path

PROJECT_ROOT = Path("..")  # Repository root.
BASELINE = PROJECT_ROOT / "data"
FIRM_DATA = PROJECT_ROOT
PATSTAT = PROJECT_ROOT / "patstat"
PATENT_TEXT = PROJECT_ROOT / "patent_text"
SAMPLEDATA = PROJECT_ROOT / "sampledata"
DATA = Path("data")          # Generated revision intermediates.
OUTPUTS = Path("outputs")    # Tables, figures and saved comparison results.
SEED = 42


# Prepare validation inputs

Prepare SI S2.2 tasks and scores for `02_judge_validation.ipynb` and `03_validation.ipynb`. Run from `revision_notebooks/` with paths set in the first cell. Save tasks and scores under `DATA/validation/`.

Read `BASELINE/candidate_pair_scores.parquet` from `../dataprep/01_fse_embeddings_and_scores.ipynb` and publication titles/abstracts. Place the human-rating files in `DATA/validation/`: `batch11.tsv` contains `ccmt_patent_id,patent_id1,patent_id2,choice1`, and `batch12.tsv` contains the same patent IDs with `choice2`. Rated patent IDs include `US`.

Keep the 200 rated triples in their recorded order, then add 800 title/abstract tasks sampled from the four-field score table.


In [ ]:
import hashlib
import html
import json
import random
from collections import defaultdict
import duckdb
import numpy as np
import pandas as pd

VALIDATION = DATA / 'validation'
VALIDATION.mkdir(parents=True, exist_ok=True)
PAIR_DATA = BASELINE / 'candidate_pair_scores.parquet'
TEXT_DATA = PATENT_TEXT / 'g_patent_title_abstract.parquet'
METRICS = ('abstract', 'summary', 'claim', 'detail', 'score')
LOW_MID, MID_HIGH = 2.915233, 3.109160
SAMPLE_SEED = 20260807
ROLES = ('focal', 'patent_1', 'patent_2')


In [ ]:
def load_tasks(path=VALIDATION / "tasks.jsonl"):
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle]

def write_tasks(tasks, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for task in tasks:
            handle.write(json.dumps(task, ensure_ascii=False, sort_keys=True) + "\n")

def task_id(source, focal, first, second):
    identity = f"{source}|{focal}|{first}|{second}"
    suffix = hashlib.sha256(identity.encode()).hexdigest()[:12]
    return f"{source}_{focal.removeprefix('US')}_{suffix}"

def category(score):
    return "high" if score > MID_HIGH else "mid" if score > LOW_MID else "low"

def human_tasks():
    """Keep the rated triples, publication IDs and presentation order."""
    human = pd.read_csv(VALIDATION / "batch11.tsv", sep="\t", dtype=str)
    tasks = []
    for row, (focal, first, second, _) in enumerate(human.itertuples(index=False, name=None), 1):
        tasks.append({
            "task_id": task_id("human", focal, first, second),
            "source": "human", "source_row": row,
            "focal_patent_id": focal, "patent_1_id": first, "patent_2_id": second,
            "sampling": {"protocol": "original_human_pair",
                         "presentation_order": "original_patent_1_patent_2_order"},
        })
    return tasks


## Additional title/abstract tasks

The 200 human triples retain their recorded order. Draw 850 extra focal patents, retain the first 800 with complete titles/abstracts and balance the top-pool candidate's position. All patent IDs here are **publications**; the score table omits the `US` prefix, while tasks include it.

In [ ]:
pair_path = PAIR_DATA
text_path = TEXT_DATA
seed = SAMPLE_SEED
n_extra = 800

import duckdb

human = human_tasks()
human_focals = {t["focal_patent_id"].removeprefix("US") for t in human}
con = duckdb.connect()
pairs = con.read_parquet(str(pair_path))
pairs.create_view("pairs")
eligible = con.sql("""
    SELECT ccmt_patent_id FROM pairs
    WHERE greatest(abstract, summary, claim, detail) < .99 AND score IS NOT NULL
    GROUP BY ccmt_patent_id HAVING count(*) > 10 ORDER BY ccmt_patent_id
""").fetchall()
eligible = [f for (f,) in eligible if f not in human_focals]
focals = random.Random(seed).sample(eligible, n_extra + max(50, int(n_extra * .05)))
con.register("selected", pd.DataFrame({"ccmt_patent_id": focals}))
rows = con.sql("""
    SELECT p.ccmt_patent_id, p.patent_id, CAST(p.score AS DOUBLE)
    FROM pairs p JOIN selected USING (ccmt_patent_id)
    WHERE greatest(abstract, summary, claim, detail) < .99 AND score IS NOT NULL
    ORDER BY p.ccmt_patent_id, p.score DESC, p.patent_id
""").fetchall()
candidates = defaultdict(list)
for focal, patent, score in rows:
    candidates[focal].append((patent, score, category(score)))
extra = []
for row, focal in enumerate(focals, 1):
    group = candidates[focal]
    focal_seed = int(hashlib.sha256(f"{seed}|{focal}".encode()).hexdigest()[:16], 16)
    rng = random.Random(focal_seed)
    top_n = max(1, int(len(group) * .1))
    first = rng.choice(group[:top_n])
    remaining = [c for c in group if c[0] != first[0]]
    available = [c for c in ("low", "mid", "high") if any(p[2] == c for p in remaining)]
    drawn = rng.choice(available)
    second = rng.choice([p for p in remaining if p[2] == drawn])
    extra.append({
        "source": "llm_extra", "source_row": row,
        "focal_patent_id": "US" + focal,
        "patent_1_id": "US" + first[0], "patent_2_id": "US" + second[0],
        "sampling": {
            "protocol": "top_10_percent_vs_global_tercile", "seed": seed,
            "focal_seed": focal_seed, "candidate_count": len(group), "top_pool_count": top_n,
            "drawn_score_category": drawn, "top_candidate_position": 1,
            "patent_1_score": first[1], "patent_1_score_category": first[2],
            "patent_2_score": second[1], "patent_2_score_category": second[2],
        },
    })
ids = sorted({t[f"{r}_patent_id" if r == "focal" else f"{r}_id"].removeprefix("US")
              for t in human + extra for r in ROLES})
con.register("requested", pd.DataFrame({"patent_id": ids}))
con.read_parquet(str(text_path)).create_view("patent_text")
texts = {}
for patent, title, abstract in con.sql("""
    SELECT patent_id, patent_title, patent_abstract FROM patent_text JOIN requested USING (patent_id)
""").fetchall():
    texts[patent] = [" ".join(html.unescape(str(s or "")).split()) for s in (title, abstract)]
con.close()
id_fields = ("focal_patent_id", "patent_1_id", "patent_2_id")
extra = [t for t in extra if all(all(texts.get(t[k].removeprefix("US"), ("", "")))
                                for k in id_fields)][:n_extra]
first_positions = set(random.Random(seed ^ 0x5A17C0DE).sample(range(len(extra)), len(extra) // 2))
for i, task in enumerate(extra):
    sampling = task["sampling"]
    if i not in first_positions:
        task["patent_1_id"], task["patent_2_id"] = task["patent_2_id"], task["patent_1_id"]
        for suffix in ("score", "score_category"):
            a, b = f"patent_1_{suffix}", f"patent_2_{suffix}"
            sampling[a], sampling[b] = sampling[b], sampling[a]
        sampling["top_candidate_position"] = 2
    task["source_row"] = i + 1
    task["task_id"] = task_id("llm_extra", *(task[k] for k in id_fields))
tasks = human + extra
for task in tasks:
    for role, key in zip(ROLES, id_fields):
        task[f"{role}_title"], task[f"{role}_abstract"] = texts[task[key].removeprefix("US")]
print(pd.Series([t["source"] for t in tasks]).value_counts())
write_tasks(tasks, VALIDATION / "tasks.jsonl")
pd.Series([t['source'] for t in tasks]).value_counts()


## Save candidate scores for the prepared tasks

Read the `tasks.jsonl` saved above and extract both candidates' scores from the same candidate-score table. `pair_scores.parquet` is saved beside the task file for notebook 03.

In [ ]:
tasks = load_tasks()

import duckdb

ids = pd.DataFrame(tasks)[["task_id", "focal_patent_id", "patent_1_id", "patent_2_id"]].copy()
for field in ids.columns[1:]:
    ids[field] = ids[field].str.removeprefix("US")
con = duckdb.connect()
con.register("requested", ids)
con.read_parquet(str(pair_path)).create_view("pairs")
fields = ", ".join(f"p{i}.{m} AS patent_{i}_{m}" for i in (1, 2) for m in METRICS)
scores = con.sql(f"""
    SELECT r.task_id, {fields} FROM requested r
    JOIN pairs p1 ON p1.ccmt_patent_id = r.focal_patent_id AND p1.patent_id = r.patent_1_id
    JOIN pairs p2 ON p2.ccmt_patent_id = r.focal_patent_id AND p2.patent_id = r.patent_2_id
""").df()
con.close()
scores.to_parquet(VALIDATION / 'pair_scores.parquet', index=False)
scores.head()


## Full-text assembly for the human-rated tasks

Assemble full text for the 200 human-rated tasks to compare title/abstract and full-text judgments. Use publication grant dates (`patent_date`) to select yearly summary, claims and description files. Join claims in `claim_sequence` order and keep each field in full. Save `DATA/validation/tasks_fulltext.jsonl` for notebook 02.


In [ ]:
def assemble_fulltext(tasks, text_path=TEXT_DATA, text_root=PATENT_TEXT):
    """Attach full summaries, ordered claims and detailed descriptions."""
    import duckdb

    keys = ("focal_patent_id", "patent_1_id", "patent_2_id")
    ids = sorted({t[k].removeprefix("US") for t in tasks for k in keys})
    con = duckdb.connect()
    con.register("requested", pd.DataFrame({"patent_id": ids}))
    con.read_parquet(str(text_path)).create_view("patent_text")
    dates = con.sql("""
        SELECT patent_id, patent_date FROM requested LEFT JOIN patent_text USING (patent_id)
    """).fetchall()
    years = {patent: int(str(date)[:4]) for patent, date in dates}
    fields = {p: {f: "" for f in ("summary", "claims", "detail")} for p in ids}
    sources = {
        "summary": ("summary/g_brf_sum_text_{year}.parquet", "summary_text"),
        "claims": ("claim/g_claims_{year}.parquet", "claim_text"),
        "detail": ("detail/g_detail_desc_text_{year}.parquet", "description_text"),
    }
    for year in sorted(set(years.values())):
        con.register("selected", pd.DataFrame({"patent_id": [p for p in ids if years[p] == year]}))
        for field, (filename, column) in sources.items():
            con.read_parquet(str(text_root / filename.format(year=year))).create_view("field_text")
            expression = "string_agg(claim_text, '\n\n' ORDER BY claim_sequence)" if field == "claims" else column
            group = " GROUP BY patent_id" if field == "claims" else ""
            for patent, value in con.sql(f"SELECT patent_id, {expression} FROM field_text JOIN selected USING (patent_id){group}").fetchall():
                fields[patent][field] = str(value or "").replace("\x00", "").strip()
    con.close()
    result = []
    for task in tasks:
        record = dict(task)
        for role, key in zip(ROLES, keys):
            patent = task[key].removeprefix("US")
            record[f"{role}_grant_year"] = years[patent]
            record.update({f"{role}_{f}": value for f, value in fields[patent].items()})
        result.append(record)
    write_tasks(result, VALIDATION / "tasks_fulltext.jsonl")
    return result


In [ ]:
human = [task for task in load_tasks() if task["source"] == "human"]
fulltext = assemble_fulltext(human)
